# Data Programming in Python | BAIS:6040
# Notebook Demonstration 7.5 Implementing a Machine Learning Algorithm

Topics to be covered:
- Fitting your machine learning model
- Evaluating performance of your model

### This code is from Notebook Demonstration 7.4. Run everything above "Using a Machine Learning Algorithm" and then begin there.

In [ ]:
# ! pip install --user --upgrade scikit-learn

## Data Loading and Preparation

### Loading Data into a Pandas Dataframe

In [ ]:
from seaborn import load_dataset

df = load_dataset("titanic")
df

### Selecting Columns of Interest

In [ ]:
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]]
df

We want to filter out unnecessary or duplicate columns. 

### Handling Missing Data

Most machine learning libraries will not accept null values as input. Every null value in a data set must be removed or replaced with a valid value. 

In [ ]:
df.info()

pandas.DataFrame.info: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html

The `age` columns has 714 non-null values, which means the other 177 values are null. 

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df = df.dropna()
df

We want to drop all rows with any missing values. Be aware that we have lost those 177 rows using this approach. 

In [ ]:
df.info()

## Supervised Learning - Binary Classification

### Setting the Goal

Using the Titanic data set, we aim to build a classification model that is able to predict whether an imaginery passenger with a certain class, sex, age, company, and fare would have survived the Titanic accident or not. This is a binary classification problem. 

For example, suppose there was a man of age 25 who purchased a third class ticket at £7 and was on board by himself, would he probably have died or survived?

In [ ]:
df.survived.value_counts()

For binary classification, we oftern refer to a *positive* class, in this case class 1, and a *negative* class, in this case class 0, with the understanding that the positive class is the one we are looking for, which is usually the minority class. 

### Defining the Features and the Target

In machine learning, the individual items are called *samples* and their properties are called *features*. In this case, we have 714 samples and 6 features. 

In [ ]:
features = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
target = "survived"

According to the goal description above, we predict `survived` using `pclass`, `sex`, `age`, `sibsp`, `parch`, and `fare`. 

In [ ]:
X = df[features]
y = df[target]

For a supervised learning task, you need a features set `X` and a target set `y`.

In [ ]:
X

`X` is a Pandas dataframe.

In [ ]:
y

`y` is a Pandas series.

In `scikit-learn`, following conventions from mathematics, data is usually denoted with an uppercase `X`, while labels are denoted by a lowercase `y`. We use an uppercase `X` because the data is a 2-dimensional array (a matrix) and a lowercase `y` because the target is a 1-dimensional array (a vector). 

### Converting Categorical Columns into Numerical Columns

As most machine learning packages will only accept numbers as input, every categorical column in a dataset must be replaced with a numerical column. 

In [ ]:
X

In [ ]:
X.info()

In [ ]:
X.sex.value_counts()

In [ ]:
X.sex = X.sex.apply(lambda x: 1 if x == "male" else 0)

pandas.Series.apply: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.apply.html

We want to convert *male* to 1 and *female* to 0. 

In [ ]:
X

Another option is to use sklearn's label_binarize function

In [ ]:
X = df[features]
y = df[target]

In [ ]:
from sklearn.preprocessing import label_binarize
X['sex'] = label_binarize(X.sex, classes=['female', 'male'])

In [ ]:
X.sex

In [ ]:
X

### Splitting Data into Training and Test data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=0)

sklearn.model_selection.train_test_split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

To assess model performance, we need to randomly split the features set `X` and the target set `y` into two training sets `X_train` and `y_train` and two test sets `X_text` and `y_test`. Here, `X_train` and `y_train` will be used for training a model, while `X_test` and `y_test` will be used for testing the model. 

Setting the `test_size` parameter to 0.25 means splitting the data into 25% of test data and 75% of training data. 

The `random_state` parameter controls the shuffling applied to the data before applying the split. You can pass an int for reproducible (deterministic) output across multiple function calls. 

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

### Using a Machine Learning Algorithm

Let's start with the k-Nearest Neigobors (k-NNs) algorithm as our first classification algorithm to try. 

### Initializing an Estimator by Setting Hyper-Parameters

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knc = KNeighborsClassifier(n_neighbors=1)
knc

class sklearn.neighbors.KNeighborsClassifier(`n_neighbors`=5, \*, `weights`='uniform', `algorithm`='auto', `leaf_size`=30, `p`=2, `metric`='minkowski', `metric_params`=None, `n_jobs`=None, \*\*kwargs)

sklearn.neighbors.KNeighborsClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

The number of neighbors `n_neighbors` is set to 1, which means it will consider only the closest neighbor. 

The initialization returns an object called estimator. The `knc` object will be used as the estimator for our k-NNs model.

### Fitting the Model to the Training Data

In [ ]:
knc.fit(X_train, y_train)

Fitting, or training, is done. 

### Evaluating the Performance of Model

Accuracy is the number of correct predictions (TP + TN) divided by the number of all samples. 

In [ ]:
knc.score(X_train, y_train)                   # Get the training set accuracy of the model 

In [ ]:
knc.score(X_test, y_test)                     # Get the test set accuracy of the model 

In [ ]:
summary = dict()

summary["k-NNs"] = round(knc.score(X_test, y_test), 3)
summary

We want to save the performance score of each algorithm in a dictionary, so that we can compare all the scores at the end. 

<hr>

### Trying Different Parameter Values

As for the the number of closest neighbors to consider, we have tried 1. Now let's try 3 this time with hopes that looking at three would work better than just one. 

In [ ]:
knc = KNeighborsClassifier(n_neighbors=3)
knc

You need to re-initialize the estimator whenever you change any hyper-parameter. 

In [ ]:
knc.fit(X_train, y_train)

In [ ]:
knc.score(X_train, y_train), knc.score(X_test, y_test)

It seems like increasing `n_neighbors` would not help with the performance, which makes us stay with the previous model. 